In [38]:
!pip install transformers seqeval evaluate datasets

# Data loading

In [39]:
from datasets import load_dataset

dataset_raw= load_dataset("lfcc/portuguese_ner") #id do repositorio do dataset # no nome do dataset no hugging face fazer copy
dataset_raw

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [40]:
dataset_raw["train"].features
#entidades que temos
#modelo iob (o nao e entidade, b é o inicio da entidade, i os tokens seguintes de data)

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

# Data Pre-Processing

In [41]:
from transformers import AutoTokenizer

#quero usar o tokenizer que foi ussado para treinar ente modelo
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-large-portuguese-cased")

In [42]:
inputs= tokenizer("As aulas de PLNEB são muito interessantes!")
inputs #parte a string em tokens e passa o para um vocabulario numerico
#maior frequencia da palavra menor vai ser o identificador
#510 é as
#6880 é aulas..


{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 19591, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [43]:
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"])#converte os ids para tokens
print(tokens)
#PLNEB é uma palavra super rara, tentou partir em subwords que ele conhece
#[CLS] token: para o treino do modelo, diz se duas frases fazem sentido ou nao
#[SEP] token: a seguir ao sep temos outra frase
#sep e cls sao uma heranca do treino do modelo, ignoramo-los

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', '##EB', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [44]:
dataset_raw["train"]["tokens"] #o dataset ja e tokens nao é string que o nosso modelo recebe

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01']])

In [45]:
tokens= ["as", "aulas", "plneb", "são", "interessantes", "!"] #em vez de ser uma string e uma lista
inputs=tokenizer(tokens, is_split_into_words=True) #percebe que e uma lista de tokens e converte na mesma para os ids

new_tokens= tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(new_tokens)

['[CLS]', 'as', 'aulas', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [46]:
len(tokens), len(new_tokens) #como recebe o new_tokens temos de mapear as labels, pois nao temos uma correspondencia de label para cada um destes tokens

(6, 10)

In [47]:
inputs.word_ids()#correspondencia direta entre os novos tokens e os tokens originais

[None, 0, 1, 2, 2, 2, 3, 4, 5, None]

In [48]:
def align_labels_with_tokens(word_ids, labels):
    new_labels=[]
    previous_word=None

    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100) #-100: codigo do bert para ignorar esta label

        elif previous_word != word_id:
            new_labels.append(labels[word_id]) #word_id ja e o indice

        else:
            new_labels.append(-100) #ignora as subwords
        previous_word=word_id
    return new_labels



def tokenize_dataset(dataset):
    res = []

    for row in dataset:
        inputs= tokenizer(row["tokens"], is_split_into_words=True, truncation= True, max_length=512)# e necessario delimitar o tamanho maximo, pois ha frases que ultrapassam o tamanho maximo do modelo que é 512
        new_labels= align_labels_with_tokens(inputs.word_ids(), row["ner_tags"])
        inputs["labels"] = new_labels
        res.append(inputs)

    return res

train_data = tokenize_dataset(dataset_raw["train"]) #listas com dados de teste e traino
test_data = tokenize_dataset(dataset_raw["test"])
print(len(train_data), len(test_data))
print(train_data[:10])

3716 930
[{'input_ids': [101, 1656, 3347, 131, 5523, 6046, 18961, 122, 4216, 10780, 151, 1479, 119, 11019, 122, 120, 291, 17642, 173, 187, 11964, 18394, 117, 1838, 1479, 117, 8852, 5892, 125, 18868, 7286, 7545, 22308, 6213, 15289, 22301, 122, 3410, 113, 291, 806, 114, 13056, 171, 5463, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, -100, 0, 7, 8, 8, 0, 7, -100, -100, 8, 0, 0, 0, -100, -100, 0, 0, 3, -100, -100, 0, 3, 4, 0, 0, 0, 0, 3, -100, -100, -100, 4, -100, -100, 0, 0, 0, 0, 0, 0, 3, 4, 4, 0, -100]}, {'input_ids': [101, 1656, 3347, 131, 9884, 12548, 122, 17495, 8451, 119, 11019, 122, 120, 291, 17642, 173, 354, 9369, 12234, 17807, 213, 13292, 18394, 6538, 117, 617, 22280, 3981, 2848

In [49]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_data) #passamos para um objeto Dataset devido a api do hugging face
test_dataset = Dataset.from_list(test_data)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


# Model Training

In [ ]:
n_labels = len(dataset_raw["train"].features["ner_tags"].feature.names) #numero de labels que dataset tem
label_list = dataset_raw["train"].features["ner_tags"].feature.names

#associar o id as labels senao ele so preve tipo LABEL_O
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

In [59]:
from transformers import AutoModelForTokenClassification
model= AutoModelForTokenClassification.from_pretrained("neuralmind/bert-large-portuguese-cased", num_labels= n_labels, id2label=id2label, label2id=label2id) #vamos buscar o modelo
#este modelo so foi treinado para conhecer o vocabulario nao sabe fazer nada

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok

In [60]:
#data_collator faz com que as batches tenham todas os mesmo tamanho, faz um padding automatico, tanto com os tokens como as labels
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval")

def compute_metrics(p): #funcao que avalia o modelo durante o treino
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [62]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="resutados",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.146507,0.057244,0.939525,0.963913,0.951562,0.983449
2,0.029614,0.062142,0.949363,0.967395,0.958294,0.984456


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1858, training_loss=0.06550650704407461, metrics={'train_runtime': 734.155, 'train_samples_per_second': 10.123, 'train_steps_per_second': 2.531, 'total_flos': 929069095587696.0, 'train_loss': 0.06550650704407461, 'epoch': 2.0})

# Interfence

In [ ]:
texto="O João trabalha na Universidade do Minho em Braga"

In [75]:
from transformers import pipeline

classifier = pipeline("ner", model=model, tokenizer=tokenizer)
classifier(texto)

[{'entity': 'B-Pessoa',
  'score': np.float32(0.52976394),
  'index': 2,
  'word': 'João',
  'start': 2,
  'end': 6},
 {'entity': 'B-Organizacao',
  'score': np.float32(0.89473903),
  'index': 5,
  'word': 'Universidade',
  'start': 19,
  'end': 31},
 {'entity': 'I-Organizacao',
  'score': np.float32(0.95667994),
  'index': 6,
  'word': 'do',
  'start': 32,
  'end': 34},
 {'entity': 'I-Organizacao',
  'score': np.float32(0.961674),
  'index': 7,
  'word': 'Minho',
  'start': 35,
  'end': 40},
 {'entity': 'B-Local',
  'score': np.float32(0.9648837),
  'index': 9,
  'word': 'Braga',
  'start': 44,
  'end': 49}]